# Multimodal Sample Discovery

This notebook demonstrates how to find samples (IGSNs) that have data from multiple experiments or modalities. By querying the `/aimdl/partition` endpoint for different data types and finding the intersection, you can identify samples suitable for cross-modal analysis.

For example: samples that have both PDV shock data AND XRD diffraction data.

In [1]:
import sys
from pathlib import Path

import requests
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import get_girder_client

## Query partitions for each data type

Use `/aimdl/partition` to get the list of IGSNs (sample identifiers) that have at least one file of each data type.

In [2]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    
    # Get IGSNs with PDV ALPSS output data (shock experiment results)
    pdv_partitions = gc.get('aimdl/partition', parameters={'dataType': 'pdv_alpss_output'})
    # Parse partition keys to extract IGSN (before the '//' separator)
    pdv_igsns = set(key.split('//')[0] for key in pdv_partitions.keys())

print(f'IGSNs with pdv_alpss_output: {len(pdv_igsns)}')

IGSNs with pdv_alpss_output: 154


In [3]:
    # Get IGSNs with XRD derived data (diffraction experiment results)
    xrd_partitions = gc.get('aimdl/partition', parameters={'dataType': 'xrd_derived'})
    # Parse partition keys to extract IGSN (before the '//' separator)
    xrd_igsns = set(key.split('//')[0] for key in xrd_partitions.keys())

print(f'IGSNs with xrd_derived: {len(xrd_igsns)}')

IGSNs with xrd_derived: 222


## Find samples with multiple data types

Calculate the intersection: which IGSNs have BOTH pdv_alpss_output AND xrd_derived data?

In [4]:
# Find intersection: IGSNs present in both datasets
multimodal_igsns = pdv_igsns & xrd_igsns

print(f'IGSNs with BOTH pdv_alpss_output AND xrd_derived: {len(multimodal_igsns)}')
print()
print('All multimodal samples:')
for igsn in sorted(multimodal_igsns):
    pdv_count = sum(1 for key in pdv_partitions.keys() if key.split('//')[0] == igsn)
    xrd_count = sum(1 for key in xrd_partitions.keys() if key.split('//')[0] == igsn)
    print(f'  {igsn}: {pdv_count} PDV partitions, {xrd_count} XRD partitions')

IGSNs with BOTH pdv_alpss_output AND xrd_derived: 101

All multimodal samples:
  JHAMAB00019-01: 1 PDV partitions, 2 XRD partitions
  JHAMAB00019-02: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-03: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-04: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-05: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-06: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-07: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-08: 1 PDV partitions, 6 XRD partitions
  JHAMAB00019-09: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-10: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-11: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-12: 1 PDV partitions, 67 XRD partitions
  JHAMAB00019-13: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-14: 1 PDV partitions, 1 XRD partitions
  JHAMAB00019-15: 1 PDV partitions, 1 XRD partitions
  JHAMAB00020-01: 1 PDV partitions, 1 XRD partitions
  JHAMAB00020-02: 1 PDV partitions, 1 XRD partitions
  JHAMAB00020-03: 1

## Venn diagram breakdown

Show which samples are exclusive to one modality vs. present in both.

In [10]:
pdv_only = pdv_igsns - xrd_igsns
xrd_only = xrd_igsns - pdv_igsns
both = pdv_igsns & xrd_igsns

print('Coverage breakdown:')
print(f'PDV only: {len(pdv_only)} samples')
print(f'XRD only: {len(xrd_only)} samples')
print(f'Both: {len(both)} samples')
print(f'Union (either PDV or XRD): {len(pdv_igsns | xrd_igsns)} samples')

Coverage breakdown:
PDV only: 53 samples
XRD only: 121 samples
Both: 101 samples
Union (either PDV or XRD): 275 samples


## Query other data type combinations

The partition endpoint works for any data type. Try different combinations:

In [6]:
# Example: Find samples with PDV traces AND experiment logs
pdv_trace_partitions = gc.get('aimdl/partition', parameters={'dataType': 'pdv_trace'})
pdv_log_partitions = gc.get('aimdl/partition', parameters={'dataType': 'pdv_experiment_log'})
pdv_trace_igsns = set(key.split('//')[0] for key in pdv_trace_partitions.keys())
pdv_log_igsns = set(key.split('//')[0] for key in pdv_log_partitions.keys())

trace_and_log = pdv_trace_igsns & pdv_log_igsns
print(f'Samples with both pdv_trace AND pdv_experiment_log: {len(trace_and_log)}')

Samples with both pdv_trace AND pdv_experiment_log: 117
